# 01 — Exploratory Data Analysis

Exploratory analysis only — **not** part of the real pipeline (that lives in
`src/`). Goal: confirm each dataset's class imbalance ratio, check for
missing values/outliers, and sanity-check feature distributions before
building the clustering + adaptive resampling modules.

Datasets covered:
1. Credit Card Fraud (binary, extreme imbalance)
2. Mammography (binary, moderate-high imbalance)
3. Yeast (multiclass imbalance)

Run `python -m src.data.ingest --dataset all` and
`python -m src.data.preprocess --dataset all` first so the processed CSVs
referenced below exist.


In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, so `import src...` works from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

DATASETS = {
    "creditcard": {"target": "Class", "type": "binary"},
    "mammography": {"target": "target", "type": "binary"},
    "yeast_multiclass": {"target": "class", "type": "multiclass"},
}


## Helper: load a dataset's processed train split

In [ ]:
def load_train(name):
    path = f"../data/processed/{name}/{name}_train.csv"
    df = pd.read_csv(path)
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    return df

dfs = {name: load_train(name) for name in DATASETS}


## 1. Class Distribution & Imbalance Ratio

This is the single most important check — confirms the imbalance ratios
documented in the project plan (~1:578 fraud, ~1:42 mammography, up to
~1:92 across yeast classes) actually hold in the data you downloaded.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, cfg) in zip(axes, DATASETS.items()):
    df = dfs[name]
    counts = df[cfg["target"]].value_counts().sort_index()
    counts.plot(kind="bar", ax=ax, color="#3B6EA5")
    ax.set_title(name)
    ax.set_xlabel(cfg["target"])
    ax.set_ylabel("count")
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

for name, cfg in DATASETS.items():
    counts = dfs[name][cfg["target"]].value_counts()
    ratio = counts.max() / counts.min()
    print(f"{name}: majority:minority ratio = {ratio:.1f} : 1")


## 2. Missing Values Check

Should be empty/zero for all three — `preprocess.py` already drops rows with missing values, but worth confirming nothing slipped through.

In [ ]:
for name, df in dfs.items():
    n_missing = df.isna().sum().sum()
    print(f"{name}: {n_missing} missing values total")


## 3. Feature Distributions

Quick histograms per dataset to spot obvious outliers or skew that might
affect clustering in Week 3-4 (KMeans is sensitive to scale/outliers,
which is why `preprocess.py` already applies StandardScaler).

In [ ]:
for name, cfg in DATASETS.items():
    df = dfs[name]
    feature_cols = [c for c in df.columns if c != cfg["target"]][:6]  # cap at 6 for readability
    df[feature_cols].hist(figsize=(14, 6), bins=30, color="#3B6EA5")
    plt.suptitle(f"{name} — feature distributions (first {len(feature_cols)} features)")
    plt.tight_layout()
    plt.show()


## 4. Correlation Heatmap — Credit Card Fraud

Most relevant for the fraud dataset since its features are anonymized PCA
components (V1-V28); worth checking none are near-duplicate/redundant
before feeding them into clustering.

In [ ]:
df = dfs["creditcard"]
feature_cols = [c for c in df.columns if c != "Class"]
corr = df[feature_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Credit Card Fraud — Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


## 5. Minority Class in 2D (PCA projection)

A first visual look at whether the minority class forms distinct
sub-clusters or is scattered — directly motivates the per-cluster adaptive
resampling approach. If the minority class is visibly clustered into
separate groups, that's the empirical justification for treating each
cluster with a different resampler instead of one global strategy.

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, cfg) in zip(axes, DATASETS.items()):
    df = dfs[name]
    feature_cols = [c for c in df.columns if c != cfg["target"]]
    X = df[feature_cols].values
    y = df[cfg["target"]].values

    coords = PCA(n_components=2, random_state=42).fit_transform(X)

    ax.scatter(coords[:, 0], coords[:, 1], c=y, cmap="viridis", s=8, alpha=0.6)
    ax.set_title(name)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()


## 6. Summary Statistics Table

Quick numeric summary per dataset, useful to paste into the report's
dataset description section.

In [ ]:
for name, df in dfs.items():
    print(f"\n=== {name} ===")
    display(df.describe().T)


## Takeaways (fill in after running)

- [ ] Confirmed imbalance ratios match the project plan for all 3 datasets
- [ ] No missing values remain post-preprocessing
- [ ] Noted any features with extreme outliers or skew that might need special handling before clustering
- [ ] Observed whether minority class forms visible sub-clusters (motivates adaptive resampling)
